In [1]:
import pandas as pd
import numpy as np
from lightgbm import LGBMClassifier
import cupy as cp 
import gc
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from tqdm import tqdm

/usr/local/lib/python3.12/dist-packages/sqlalchemy/orm/query.py:195: SyntaxWarning: "is not" with 'tuple' literal. Did you mean "!="?
  if entities is not ():


In [2]:
df = pd.read_csv('../input/data-okx/merge.csv')
df["timestamp_dt"] = pd.to_datetime(df["timestamp_dt"], utc=True)
df = df.sort_values("timestamp_dt").reset_index(drop=True)
df = df[~df["timestamp_dt"].dt.date.isin([
        pd.to_datetime("2025-12-27").date(),
        pd.to_datetime("2025-12-28").date(),
        pd.to_datetime("2025-12-29").date()
    ])
]
df = df[df["timestamp_dt"].dt.month != 1]
df["perpetual_fundingRate_funding_rate"] = df["perpetual_fundingRate_funding_rate"].ffill()
print(df.shape)

(94752, 1383)


In [3]:
# Initial features:
#     indexPriceKlines/markPriceKline: open, high, low, close, mean, std
#     perpetual/spot trades: 
#         volume(buy, sell) 
#         turnover(buy, sell) 
#         count(buy, sell, trade, tick_up, tick_down) 
#         price(first, last, mean, std, min, max, 0,25, 0,50, 0,75, skew, kurtosis, nunique)(buy, sell, trade)
#         size(first, last, mean, std, max, 0,25, 0,50, 0,75, skew, kurtosis, nunique)(buy, sell, trade)
#         rate(mean, std, max)(buy, sell, trade) 
#         corr(price, size | price, time | size, time)
#     orderBook:
#         wmp(first, last, mean, std, min, max, 0,25, 0,50, 0,75, skew, kurtosis)
#         spread(first, last, mean, std, max, skew, kurtosis) 
#         imbal(first, last, mean, std, min, max, 0,25, 0,50, 0,75, skew, kurtosis) 
#         depth(first, last, mean, std, min, max) 
#         conc(mean, std, min, max)
#         impact(mean, std, min, max)(bid, ask)
#         pressure(sum, first, last, mean, std, min, max, 0,25, 0,50, 0,75, skew, kurtosis) 
#         count(snapshot) 
#         rate(mean, std, min, max) 
#         corr(wmp_imbal | wmp_depth | wmp_pressure | wmp_time | spread_imbal | spread_impact | pressure_time) 
#     funding_rate

# Log return
#     indexPriceKlines/markPriceKline: open, high, low, close, mean
#     perpetual/spot trades: price(first, last, mean, min, max, 0,25, 0,50, 0,75)(trade, buy, sell)
#     orderBook: wmp(first, last, mean, min, max, 0,25, 0,50, 0,75)

# Delta 
#     indexPriceKlines/markPriceKline: std
#     perpetual/spot trades: 
#         price(std)(trade, buy, sell)
#         size(mean, std)(trade, buy, sell)
#         volume(buy, sell)
#         turnover(buy, sell)
#         count(buy, sell, trade, tick_up, tick_down)
#         rate(mean)(trade, buy, sell)
#     orderBook: 
#         wmp(std)
#         spread(mean)
#         imbal(mean)
#         depth(mean)
#         conc(mean)
#         pressure(sum, mean, std)
#         count(snapshot) 
#         rate(mean)
#         impact(bid, ask)(mean)
#     funding_rate

# Lag return
#     indexPriceKlines/markPriceKline: (log return)close, (delta)std
#     perpetual/spot trades: 
#         (log return)price(mean)(trade, buy, sell), (delta)price(std)(trade, buy, sell)
#         (delta)size(mean, std)(trade, buy, sell)
#         (delta)volume(buy, sell)
#         (delta)count(buy, sell, trade, tick_up, tick_down)
#         (delta)rate(mean)(trade, buy, sell)
#     orderBook: 
#         (log return)wmp_last, (delta)wmp_std
#         (delta)depth(mean)
#         (delta)conc(mean)
#         (delta)spread(mean)
#         (delta)imbal(mean)
#         (delta)pressure(mean, std, sum)
#         (delta)count(snapshot)
#         (delta)rate(mean)
#         (delta)impact(bid, ask)(mean)

# Return Dynamics, Trend, Regime & Divergence
#     Dynamics: Rolling Mean of Return; Rolling Std of Return; Momentum; Return Acceleration
#         indexPriceKlines/markPriceKline: (log return)close
#         perpetual/spot trades: (log return)price(mean)(trade, buy, sell)
#         orderBook: (log return)wmp(last)
#     Trend:
#         Trend Strength: Log return; Rolling Std of Return
#         MA divergence (z-log return): Log return; Rolling mean of return; Rolling std of return
#         Trend vs Chop Proxy: Log return; Rolling std of return
#     Regime:
#         Efficiency Ratio: log return
#         Volatility Ratio: Rolling Std of Return
#         Z-volatility
#     Divergence: (log return)wmp_last; (log return)index_close 

# Market Microstructure Features
#     Candle Range:
#         indexPriceKlines/markPriceKline: high, low, mean
#         perpetual/spot trades: price(mean,min, max)(trade)
#     Statistical Normalization
#         z-volume: perpetual/spot trades: volume(buy, sell, total) 
#         z-spread: orderBook: spread(mean)
#         z-imbalance: orderBook: imbal(mean)
#     Order Flow (Trades)
#         Volume Imbalance: perpetual/spot trades: volume(buy, sell)
#         Count Imbalance: perpetual/spot trades: count(buy, sell)
#         Aggressiveness Ratio: perpetual/spot trades: count(buy, sell)
#         Turnover Imbalance: perpetual/spot trades: turnover(buy, sell)
#         Tick Direction Imbalance: perpetual/spot trades: count(tick_up, tick_down)
#         Flow Pressure Ratio = volume_imbalance / (depth_mean + eps)
#     Orderbook Pressure & Liquidity
#         Depth Imbalance: orderBook: depth(mean), imbal(mean)
#         Liquidity Consumption: perpetual/spot trades: volume(total); orderBook: depth(mean)
#         Slope Proxy: orderBook: spread(mean); depth(mean)
#     Spread & Transaction Cost
#         Relative Spread: orderBook.spread_mean; orderBook.wmp_mean
#         Liquidity Density: orderBook.depth_mean; orderBook.spread_mean
#         Impact Cost: impact_mean_bid; impact_mean_ask
#     Basis & Premium (Futures vs Spot)
#         Basis Spread: indexPriceKlines/markPriceKline: close
#         Basis Ratio: indexPriceKlines/markPriceKline: close
#         Premium Index: indexPriceKlines: close; orderBook: wmp(last)
#         Basis Momentum: Basis Ratio; Basis Spread
#     Funding & Sentiment
#         Funding Cost: funding_rate; mark_price_close
#         Funding × Basis (SENTIMENT INTERACTION): funding_rate; basis_ratio
#     Imbalance Divergence: volume_imbalance (trades), depth_imbalance (orderbook)

# FEATURE ENGINEERING

## Feature Generation

In [4]:
drop_cols = []
horizons = {
    "5m": 1,
    "15m": 3,
    "1h": 12,
    "4h": 48,
    "1d": 288
}
lag_map = {
    "5m": [1, 2, 3, 4, 5],
    "15m": [1, 2, 3, 4],
    "1h": [1, 2, 3],
    "4h": [1, 2],
    "1d": [1],
}
epsilon = 1e-9
action_type = ["trade", "buy", "sell"]

In [5]:
table_types = ["spot_trades", "perpetual_trades"]
for t_type in table_types:
    for hz, offset in horizons.items():
        df[f"{t_type}_{hz}_volume_trade"] = df[f"{t_type}_{hz}_volume_buy"] + df[f"{t_type}_{hz}_volume_sell"]
        df[f"{t_type}_{hz}_turnover_trade"] = df[f"{t_type}_{hz}_turnover_buy"] + df[f"{t_type}_{hz}_turnover_sell"]
        df[f"{t_type}_{hz}_count_trade"] = df[f"{t_type}_{hz}_trade_count"]
        drop_cols.append(f"{t_type}_{hz}_trade_count")

##### Log return

In [6]:
new_cols = {}   
data_dict = {
    "indexPriceKlines":             ["open", "high", "low", "close", "mean"],
    "perpetual_markPriceKlines":    ["open", "high", "low", "close", "mean"],
    "spot_trades":                  ["price_first", "price_last", "price_mean", "price_min", "price_max", "price_0.25", "price_0.50", "price_0.75"],
    "perpetual_trades":             ["price_first", "price_last", "price_mean", "price_min", "price_max", "price_0.25", "price_0.50", "price_0.75"],
    "perpetual_orderBook":          ["wmp_first", "wmp_last", "wmp_mean", "wmp_min", "wmp_max", "wmp_0.25", "wmp_0.50", "wmp_0.75"]
}
for t_type, data_types in data_dict.items():
    for hz, offset in horizons.items():
        for d_type in data_types:
            if t_type in ["spot_trades", "perpetual_trades"]:
                for a_type in action_type:
                    source_col = f"{t_type}_{hz}_{d_type}_{a_type}"
                    new_col = f"log_return_{source_col}"
                    new_cols[new_col] = np.log(df[source_col] / df[source_col].shift(offset))
                    drop_cols.append(source_col)
                    if d_type == "price_mean":
                        for lag in lag_map[hz]:
                            lag_col = f"lag_{lag}_{new_col}"
                            new_cols[lag_col] = new_cols[new_col].shift(lag*offset)
            else:
                source_col = f"{t_type}_{hz}_{d_type}"
                new_col = f"log_return_{source_col}"
                new_cols[new_col] = np.log(df[source_col] / df[source_col].shift(offset))
                drop_cols.append(source_col)
                if d_type in ["close", "wmp_last"]:
                    for lag in lag_map[hz]:
                        lag_col = f"lag_{lag}_{new_col}"
                        new_cols[lag_col] = new_cols[new_col].shift(lag*offset)

df = pd.concat(
    [df, pd.DataFrame(new_cols, index=df.index)],
    axis=1
)

##### Delta


In [7]:
new_cols = {}   
data_dict = {
    "indexPriceKlines":             ["std"],
    "perpetual_markPriceKlines":    ["std"],
    "spot_trades":                  ["price_std", "size_mean", "size_std", "volume", "turnover", "count", "rate_mean_ms"],
    "perpetual_trades":             ["price_std", "size_mean", "size_std", "volume", "turnover", "count", "rate_mean_ms"],
    "perpetual_orderBook":          ["wmp_std", "spread_mean", "imbal_mean", "depth_mean", "conc_mean", "pressure_sum", "pressure_mean", "pressure_std", "snapshot_count", "rate_mean_ms", "impact_bid_mean", "impact_ask_mean"],
    "perpetual_fundingRate":        ["funding_rate"]
}
for t_type, data_types in data_dict.items():
    if t_type == "perpetual_fundingRate":
        source_col = f"{t_type}_{data_types[0]}"
        new_col = f"delta_{source_col}"
        new_cols[new_col] = df[source_col] - df[source_col].shift(96)
    else:
        for hz, offset in horizons.items():
            for d_type in data_types:
                if t_type in ["spot_trades", "perpetual_trades"]:
                    if d_type== "count":
                        for a_type in action_type + ["tick_up", "tick_down"]:
                            source_col = f"{t_type}_{hz}_{d_type}_{a_type}"
                            new_col = f"delta_{source_col}"
                            new_cols[new_col] = df[source_col] - df[source_col].shift(offset)
                            for lag in lag_map[hz]:
                                lag_col = f"lag_{lag}_{new_col}"
                                new_cols[lag_col] = new_cols[new_col].shift(lag*offset)
                    else:
                        for a_type in action_type:
                            source_col = f"{t_type}_{hz}_{d_type}_{a_type}"
                            new_col = f"delta_{source_col}"
                            new_cols[new_col] = df[source_col] - df[source_col].shift(offset)
                            if d_type != "turnover": 
                                for lag in lag_map[hz]:
                                    lag_col = f"lag_{lag}_{new_col}"
                                    new_cols[lag_col] = new_cols[new_col].shift(lag*offset)                         
                else:
                    source_col = f"{t_type}_{hz}_{d_type}"
                    new_col = f"delta_{source_col}"
                    new_cols[new_col] = df[source_col] - df[source_col].shift(offset)
                    for lag in lag_map[hz]:
                        lag_col = f"lag_{lag}_{new_col}"
                        new_cols[lag_col] = new_cols[new_col].shift(lag*offset)
                    
df = pd.concat(
    [df, pd.DataFrame(new_cols, index=df.index)],
    axis=1
)

##### Return Dynamics, Trend & Regime

In [8]:
new_cols = {}
data_dict = {
    "log_return_indexPriceKlines":          "close",
    "log_return_perpetual_markPriceKlines": "close",
    "log_return_spot_trades":               "price_mean",
    "log_return_perpetual_trades":          "price_mean",
    "log_return_perpetual_orderBook":       "wmp_last",
}
for hz, offset in horizons.items():
    for t_type, d_type in data_dict.items():
        if "spot_trades"  in t_type or "perpetual_trades" in t_type:
            for a_type in action_type:
                source_col = f"{t_type}_{hz}_{d_type}_{a_type}"
                for window in [12, 48]:
                    # Dynamics
                    new_cols[f"rolling_mean_{window}_{source_col}"] = df[source_col].rolling(window * offset).mean().shift(1)
                    new_cols[f"rolling_std_{window}_{source_col}"] = df[source_col].rolling(window * offset).std().shift(1)
                    new_cols[f"momentum_{window}_{source_col}"] = df[source_col] - df[source_col].shift(window * offset)
                    new_cols[f"acceleration_{window}_{source_col}"] = ((df[source_col] - df[source_col].shift(window * offset)) - (df[source_col].shift(window * offset) - df[source_col].shift(2 * window * offset)))   
                    # Trend
                    new_cols[f"trend_strength_{window}_{source_col}"] =  abs(df[source_col]) / (new_cols[f"rolling_std_{window}_{source_col}"] + epsilon)
                    new_cols[f"ma_divergence_{window}_{source_col}"] =  (df[source_col] - new_cols[f"rolling_mean_{window}_{source_col}"]) / (new_cols[f"rolling_std_{window}_{source_col}"] + epsilon)
                    new_cols[f"trend_vs_chop_{window}_{source_col}"] = (abs(new_cols[f"rolling_mean_{window}_{source_col}"]) / (new_cols[f"rolling_std_{window}_{source_col}"] + epsilon))
                # Regime
                new_cols[f"volatility_ratio_{source_col}"] = new_cols[f"rolling_std_12_{source_col}"]/ (new_cols[f"rolling_std_48_{source_col}"] + epsilon)
                new_cols[f"z_volatility_{source_col}"] = (new_cols[f"rolling_std_12_{source_col}"]- new_cols[f"rolling_std_12_{source_col}"].rolling(48 * offset).mean().shift(1)) / (new_cols[f"rolling_std_12_{source_col}"].rolling(48 * offset).std().shift(1) + epsilon)
                new_cols[f"efficiency_ratio_{source_col}"] = (df[source_col].rolling(12 * offset).sum().abs() / (df[source_col].abs().rolling(12 * offset).sum() + epsilon)).shift(1)
        else:
            source_col = f"{t_type}_{hz}_{d_type}"
            for window in [12, 48]:
                # Dynamics
                new_cols[f"rolling_mean_{window}_{source_col}"] = df[source_col].rolling(window * offset).mean().shift(1)
                new_cols[f"rolling_std_{window}_{source_col}"] = df[source_col].rolling(window * offset).std().shift(1)
                new_cols[f"momentum_{window}_{source_col}"] = df[source_col] - df[source_col].shift(window * offset)
                new_cols[f"acceleration_{window}_{source_col}"] = ((df[source_col] - df[source_col].shift(window * offset)) - (df[source_col].shift(window * offset) - df[source_col].shift(2 * window * offset)))   
                # Trend
                new_cols[f"trend_strength_{window}_{source_col}"] =  abs(df[source_col]) / (new_cols[f"rolling_std_{window}_{source_col}"] + epsilon)
                new_cols[f"ma_divergence_{window}_{source_col}"] =  (df[source_col] - new_cols[f"rolling_mean_{window}_{source_col}"]) / (new_cols[f"rolling_std_{window}_{source_col}"] + epsilon)
                new_cols[f"trend_vs_chop_{window}_{source_col}"] = (abs(new_cols[f"rolling_mean_{window}_{source_col}"]) / (new_cols[f"rolling_std_{window}_{source_col}"] + epsilon))
            # Regime
            new_cols[f"volatility_ratio_{source_col}"] = new_cols[f"rolling_std_12_{source_col}"]/ (new_cols[f"rolling_std_48_{source_col}"] + epsilon)
            new_cols[f"z_volatility_{source_col}"] = (new_cols[f"rolling_std_12_{source_col}"]- new_cols[f"rolling_std_12_{source_col}"].rolling(48 * offset).mean().shift(1)) / (new_cols[f"rolling_std_12_{source_col}"].rolling(48 * offset).std().shift(1) + epsilon)
            new_cols[f"efficiency_ratio_{source_col}"] = (df[source_col].rolling(12 * offset).sum().abs() / (df[source_col].abs().rolling(12 * offset).sum() + epsilon)).shift(1)
    # Return Divergence
    new_cols[f"return_divergence_{hz}"] = (df[f"log_return_perpetual_orderBook_{hz}_wmp_last"] - df[f"log_return_indexPriceKlines_{hz}_close"])
df = pd.concat(
    [df, pd.DataFrame(new_cols, index=df.index)],
    axis=1
)

##### Market Microstructure Features

In [9]:
new_cols = {}   
table_types = ["indexPriceKlines", "perpetual_markPriceKlines", "spot_trades", "perpetual_trades", "perpetual_orderBook"]
for hz, offset in horizons.items():
    for t_type in table_types:
        if t_type in ["spot_trades", "perpetual_trades"]:
            # Candle Range
            new_cols[f"candle_range_{t_type}_{hz}_trade"] = (df[f"{t_type}_{hz}_price_max_trade"] - df[f"{t_type}_{hz}_price_min_trade"]) / (df[f"{t_type}_{hz}_price_mean_trade"] + epsilon)
            # Statistical Normalization
            for a_type in action_type:
                volume = f"{t_type}_{hz}_volume_{a_type}"
                new_col = f"z_volume_{volume}"
                new_cols[new_col] = (df[volume] - df[volume].rolling(48 * offset).mean().shift(1)) /   (df[volume].rolling(48 * offset).std().shift(1) + epsilon)
            # Order Flow (Trades)
            new_cols[f"volume_imbalance_{t_type}_{hz}"] = (df[f"{t_type}_{hz}_volume_buy"] - df[f"{t_type}_{hz}_volume_sell"]) /  (df[f"{t_type}_{hz}_volume_buy"] + df[f"{t_type}_{hz}_volume_sell"] + epsilon)
            new_cols[f"count_imbalance_{t_type}_{hz}"] = (df[f"{t_type}_{hz}_count_buy"] - df[f"{t_type}_{hz}_count_sell"]) /  (df[f"{t_type}_{hz}_count_buy"] + df[f"{t_type}_{hz}_count_sell"] + epsilon)
            new_cols[f"turnover_imbalance_{t_type}_{hz}"] = (df[f"{t_type}_{hz}_turnover_buy"] - df[f"{t_type}_{hz}_turnover_sell"]) /  (df[f"{t_type}_{hz}_turnover_buy"] + df[f"{t_type}_{hz}_turnover_sell"] + epsilon)
            new_cols[f"tick_imbalance_{t_type}_{hz}"] = (df[f"{t_type}_{hz}_count_tick_up"] - df[f"{t_type}_{hz}_count_tick_down"]) /  (df[f"{t_type}_{hz}_count_tick_up"] + df[f"{t_type}_{hz}_count_tick_down"] + epsilon)
            new_cols[f"aggressiveness_log_{t_type}_{hz}"] = np.log1p(df[f"{t_type}_{hz}_count_buy"]) -  np.log1p(df[f"{t_type}_{hz}_count_sell"])
            new_cols[f"flow_pressure_ratio_{t_type}_{hz}"] = new_cols[f"volume_imbalance_{t_type}_{hz}"] / (df[f"perpetual_orderBook_{hz}_depth_mean"] + epsilon)
        
        elif t_type in ["indexPriceKlines", "perpetual_markPriceKlines"]:
            # Candle Range
            new_cols[f"candle_range_{t_type}_{hz}"] = (df[f"{t_type}_{hz}_high"] - df[f"{t_type}_{hz}_low"]) / (df[f"{t_type}_{hz}_mean"] + epsilon)
        
        elif t_type == "perpetual_orderBook":
            # Statistical Normalization
            for d_type in ["spread", "imbal"]:
                source_col = f"{t_type}_{hz}_{d_type}_mean"
                new_col = f"z_{d_type}_{source_col}"
                new_cols[new_col] = (df[source_col] - df[source_col].rolling(48 * offset).mean().shift(1)) /   (df[source_col].rolling(48 * offset).std().shift(1) + epsilon)
            # Orderbook Pressure & Liquidity
            new_cols[f"depth_imbalance_{t_type}_{hz}"] = df[f"{t_type}_{hz}_depth_mean"] * df[f"{t_type}_{hz}_imbal_mean"].abs()
            new_cols[f"impact_imbalance_{t_type}_{hz}"] = (df[f"{t_type}_{hz}_impact_bid_mean"] - df[f"{t_type}_{hz}_impact_ask_mean"]) /  (df[f"{t_type}_{hz}_impact_bid_mean"] + df[f"{t_type}_{hz}_impact_ask_mean"] + epsilon)
            new_cols[f"liquidity_consumption_spot_trades_{hz}"] = df[f"spot_trades_{hz}_volume_trade"] / (df[f"{t_type}_{hz}_depth_mean"] + epsilon)
            new_cols[f"liquidity_consumption_perpetual_trades_{hz}"] = df[f"perpetual_trades_{hz}_volume_trade"] / (df[f"{t_type}_{hz}_depth_mean"] + epsilon)
            new_cols[f"slope_proxy_{t_type}_{hz}"] = df[f"{t_type}_{hz}_depth_mean"] / (df[f"{t_type}_{hz}_spread_mean"] + epsilon)
            # Spread & Transaction Cost
            new_cols[f"relative_spread_{t_type}_{hz}"] = df[f"{t_type}_{hz}_spread_mean"] / (df[f"{t_type}_{hz}_wmp_mean"] + epsilon)
            new_cols[f"liquidity_density_{t_type}_{hz}"] = df[f"{t_type}_{hz}_depth_mean"] / (df[f"{t_type}_{hz}_spread_mean"] + epsilon)
            new_cols[f"impact_cost_{t_type}_{hz}"] = 0.5 * (df[f"{t_type}_{hz}_impact_bid_mean"] + df[f"{t_type}_{hz}_impact_ask_mean"])
    # Basis & Premium (Futures vs Spot)        
    new_cols[f"basis_spread_{hz}"] = df[f"perpetual_markPriceKlines_{hz}_close"]- df[f"indexPriceKlines_{hz}_close"]
    new_cols[f"basis_ratio_{hz}"] = (df[f"perpetual_markPriceKlines_{hz}_close"] - df[f"indexPriceKlines_{hz}_close"]) / (df[f"indexPriceKlines_{hz}_close"] + epsilon)
    new_cols[f"premium_index_{hz}"] = (df[f"perpetual_orderBook_{hz}_wmp_last"] - df[f"indexPriceKlines_{hz}_close"]) / (df[f"indexPriceKlines_{hz}_close"] + epsilon)
    new_cols[f"basis_momentum_{hz}"] = (new_cols[f"basis_ratio_{hz}"]- new_cols[f"basis_ratio_{hz}"].shift(12 * offset))    
    # Funding & Sentiment
    new_cols[f"funding_cost_{hz}"] = df["perpetual_fundingRate_funding_rate"] * df[f"perpetual_markPriceKlines_{hz}_close"]
    new_cols[f"funding_basis_interaction_{hz}"] = df["perpetual_fundingRate_funding_rate"] * new_cols[f"basis_ratio_{hz}"]
    # Imbalance Divergence
    new_cols[f"imbalance_divergence_{hz}"] = (new_cols[f"volume_imbalance_spot_trades_{hz}"] - new_cols[f"z_imbal_perpetual_orderBook_{hz}_imbal_mean"])
df = pd.concat(
    [df, pd.DataFrame(new_cols, index=df.index)],
    axis=1
)

##### Time & Seasonality

In [10]:
new_cols = {}
df["timestamp_dt"] = df["timestamp_dt"].dt.tz_localize(None)
hour = df["timestamp_dt"].dt.hour
minute = df["timestamp_dt"].dt.minute
month = df["timestamp_dt"].dt.month
new_cols["feat_hour_sin"] = np.sin(2 * np.pi * hour / 24)
new_cols["feat_hour_cos"] = np.cos(2 * np.pi * hour / 24)
new_cols["feat_minute_sin"] = np.sin(2 * np.pi * minute / 60)
new_cols["feat_minute_cos"] = np.cos(2 * np.pi * minute / 60)
new_cols["month"] = month
df = pd.concat(
    [df, pd.DataFrame(new_cols, index=df.index)],
    axis=1
)

##### Label

In [11]:
df = df.sort_values("timestamp_dt").reset_index(drop=True)
window = 24
alpha = 0.5
fee_threshold = 0.0012

df["ret_fwd_5m"] = (df["indexPriceKlines_5m_close"].shift(-1) - df["indexPriceKlines_5m_close"]) / df["indexPriceKlines_5m_close"]
df["vol_5m"] = (df["ret_fwd_5m"].shift(1).rolling(window).std())
df["dynamic_threshold"] = (alpha * df["vol_5m"]) #+ fee_threshold
df["label_5m"] = np.select(
    [
        df["ret_fwd_5m"] >  df["dynamic_threshold"], 
        df["ret_fwd_5m"] < -df["dynamic_threshold"], 
    ],
    [2, 1],
    default=0,
)

df["ret_lag1"] = df["ret_fwd_5m"].shift(1) 
df["ret_lag2"] = df["ret_fwd_5m"].shift(2) 
df["ret_lag3"] = df["ret_fwd_5m"].shift(3) 
df["trend_mean"] = df["ret_fwd_5m"].shift(1).rolling(window).mean()
df["return_zscore"] = (df["ret_lag1"] - df["trend_mean"]) / (df["vol_5m"] + 1e-9)
drop_cols += ["ret_fwd_5m"]

In [12]:
temp = df.iloc[::1].copy()

def label_persistence(labels: pd.Series) -> float:
    y = labels.dropna()
    if len(y) < 2:
        return np.nan
    return (y.iloc[1:].values == y.iloc[:-1].values).mean()

def neutral_ratio(labels: pd.Series) -> float:
    y = labels.dropna()
    return (y == 0).mean()

def up_ratio(labels: pd.Series) -> float:
    y = labels.dropna()
    return (y == 2).mean()

def down_ratio(labels: pd.Series) -> float:
    y = labels.dropna()
    return (y == 1).mean()

def signal_to_noise(returns: pd.Series) -> float:
    r = returns.dropna()
    if r.std() == 0:
        return np.nan
    return np.abs(r.mean()) / r.std()

metrics = pd.DataFrame(
    [{
        "horizon": "5m",
        "window": 24,
        "label_persistence": label_persistence(temp["label_5m"]),
        "neutral_ratio": neutral_ratio(temp["label_5m"]),
        "up_ratio": up_ratio(temp["label_5m"]),
        "down_ratio": down_ratio(temp["label_5m"]),
        "signal_to_noise": signal_to_noise(temp["ret_fwd_5m"]),
        "sample_size": temp["label_5m"].notna().sum(),
    }]
)

metrics

,horizon,window,label_persistence,neutral_ratio,up_ratio,down_ratio,signal_to_noise,sample_size
0,5m,24,0.351068,0.42378,0.287656,0.288564,0.000544,94752


## Feature Selection

##### Unique, variance

In [13]:
drop_cols = (drop_cols + 
[   "perpetual_fundingRate_funding_time",
    "spot_trades_5m_last_trade_time_ms", 
    "spot_trades_15m_last_trade_time_ms", 
    "spot_trades_1h_last_trade_time_ms",
    "spot_trades_4h_last_trade_time_ms", 
    "spot_trades_1d_last_trade_time_ms",
    "perpetual_trades_5m_last_trade_time_ms", 
    "perpetual_trades_15m_last_trade_time_ms", 
    "perpetual_trades_1h_last_trade_time_ms",
    "perpetual_trades_4h_last_trade_time_ms", 
    "perpetual_trades_1d_last_trade_time_ms",
    "perpetual_orderBook_5m_last_update_time_ms", 
    "perpetual_orderBook_15m_last_update_time_ms",
    "perpetual_orderBook_1h_last_update_time_ms", 
    "perpetual_orderBook_4h_last_update_time_ms",
    "perpetual_orderBook_1d_last_update_time_ms", 
    
    "spot_trades_5m_rate_min_ms_trade",
    "spot_trades_5m_rate_min_ms_sell",
    "spot_trades_5m_rate_min_ms_buy",
    "spot_trades_15m_rate_min_ms_trade",
    "spot_trades_15m_rate_min_ms_buy",
    "spot_trades_15m_rate_min_ms_sell",
    "spot_trades_1h_rate_min_ms_trade",
    "spot_trades_1h_rate_min_ms_buy",
    "spot_trades_1h_rate_min_ms_sell",
    "spot_trades_4h_rate_min_ms_trade",
    "spot_trades_4h_rate_min_ms_buy",
    "spot_trades_4h_rate_min_ms_sell",
    "spot_trades_1d_rate_min_ms_trade",
    "spot_trades_1d_rate_min_ms_sell",
    "spot_trades_1d_rate_min_ms_buy",
    
    "perpetual_trades_5m_rate_min_ms_buy",
    "perpetual_trades_5m_rate_min_ms_sell",
    "perpetual_trades_5m_rate_min_ms_trade",
    "perpetual_trades_15m_rate_min_ms_buy",
    "perpetual_trades_15m_rate_min_ms_sell",
    "perpetual_trades_15m_rate_min_ms_trade",
    "perpetual_trades_1h_rate_min_ms_buy",
    "perpetual_trades_1h_rate_min_ms_sell",
    "perpetual_trades_1h_rate_min_ms_trade",
    "perpetual_trades_4h_rate_min_ms_buy",
    "perpetual_trades_4h_rate_min_ms_sell",
    "perpetual_trades_4h_rate_min_ms_trade",
    "perpetual_trades_1d_rate_min_ms_buy",
    "perpetual_trades_1d_rate_min_ms_sell",
    "perpetual_trades_1d_rate_min_ms_trade",


    "perpetual_orderBook_5m_spread_min",
    "perpetual_orderBook_5m_spread_0.25",
    "perpetual_orderBook_5m_spread_0.50",
    "perpetual_orderBook_5m_spread_0.75",
    "perpetual_orderBook_15m_spread_min",
    "perpetual_orderBook_15m_spread_0.25",
    "perpetual_orderBook_15m_spread_0.50",
    "perpetual_orderBook_15m_spread_0.75",
    "perpetual_orderBook_1h_spread_min",
    "perpetual_orderBook_1h_spread_0.25",
    "perpetual_orderBook_1h_spread_0.50",
    "perpetual_orderBook_1h_spread_0.75",
    "perpetual_orderBook_4h_spread_min",
    "perpetual_orderBook_4h_spread_0.25",
    "perpetual_orderBook_4h_spread_0.50",
    "perpetual_orderBook_4h_spread_0.75",
    "perpetual_orderBook_1d_spread_min",
    "perpetual_orderBook_1d_spread_0.25",
    "perpetual_orderBook_1d_spread_0.50",
    "perpetual_orderBook_1d_spread_0.75",
    
    "spot_trades_1d_size_min_trade",
    "spot_trades_1d_size_min_buy",
    "spot_trades_1d_size_min_sell",
    "spot_trades_4h_size_min_trade",
    "spot_trades_4h_size_min_buy",
    "spot_trades_4h_size_min_sell",
    "spot_trades_1h_size_min_trade",
    "spot_trades_1h_size_min_buy",
    "spot_trades_1h_size_min_sell",
    "spot_trades_15m_size_min_trade",
    "spot_trades_15m_size_min_buy",
    "spot_trades_15m_size_min_sell",
    "spot_trades_5m_size_min_trade",
    "spot_trades_5m_size_min_buy",
    "spot_trades_5m_size_min_sell",
    
    "perpetual_trades_1d_size_min_trade",
    "perpetual_trades_1d_size_min_buy",
    "perpetual_trades_1d_size_min_sell",
    "perpetual_trades_4h_size_min_trade",
    "perpetual_trades_4h_size_min_buy",
    "perpetual_trades_4h_size_min_sell",
    "perpetual_trades_1h_size_min_trade",
    "perpetual_trades_1h_size_min_buy",
    "perpetual_trades_1h_size_min_sell",
    "perpetual_trades_15m_size_min_trade",
    "perpetual_trades_15m_size_min_buy",
    "perpetual_trades_15m_size_min_sell",
    "perpetual_trades_5m_size_min_trade",
    "perpetual_trades_5m_size_min_buy",
    "perpetual_trades_5m_size_min_sell",
    
    "perpetual_orderBook_1d_ts_ms",
    "perpetual_orderBook_4h_ts_ms",
    "perpetual_orderBook_1h_ts_ms",
    "perpetual_orderBook_15m_ts_ms",
    "perpetual_orderBook_5m_ts_ms",
    "perpetual_trades_1d_ts_ms",
    "perpetual_trades_4h_ts_ms",
    "perpetual_trades_1h_ts_ms",
    "perpetual_trades_15m_ts_ms",
    "perpetual_trades_5m_ts_ms",
    "spot_trades_1d_ts_ms",
    "spot_trades_4h_ts_ms",
    "spot_trades_1h_ts_ms",
    "spot_trades_15m_ts_ms",
    "spot_trades_5m_ts_ms",
    "perpetual_markPriceKlines_1d_ts_ms",
    "perpetual_markPriceKlines_4h_ts_ms",
    "perpetual_markPriceKlines_1h_ts_ms",
    "perpetual_markPriceKlines_15m_ts_ms",
    "perpetual_markPriceKlines_5m_ts_ms",
    "indexPriceKlines_1d_ts_ms",
    "indexPriceKlines_4h_ts_ms",
    "indexPriceKlines_1h_ts_ms",
    "indexPriceKlines_15m_ts_ms",
    "indexPriceKlines_5m_ts_ms"])
df.drop(columns=drop_cols, errors="ignore",inplace=True)

In [14]:
# nuniques = df.nunique().sort_values(ascending=True)
# vars_ = df.var(numeric_only=True)
# for col_name, cnt in nuniques.items():
#         var = vars_.get(col_name, np.nan)
#         print(f"{col_name}: {cnt} uniques | var={var}")

In [15]:
# vars_ = vars_.sort_values(ascending=True)
# for col_name, var in vars_.items():
#         cnt = nuniques.get(col_name, np.nan)
#         print(f"{col_name}: {cnt} uniques | var={var}")

##### Correlation

In [16]:
# #---------------------------------------------------------------------------------------------#
# #---------------------------------------------------------------------------------------------#
# #---------------------------------------------------------------------------------------------#
# print("🔹 [1/4] Preparing data...")
# # Load và ép kiểu ngay lập tức
# train_df = df[df["month"].isin(list(range(2, 12)))].drop(columns=["timestamp_dt", "month"])
# X = train_df.drop(columns=["label_5m"]).astype(np.float32)
# y = train_df["label_5m"]
# cols = X.columns.to_numpy()
# del train_df
# gc.collect()
# #---------------------------------------------------------------------------------------------#
# #---------------------------------------------------------------------------------------------#
# #---------------------------------------------------------------------------------------------#
# print("🔹 [2/4] Computing correlation matrix (GPU)...")
# CORR_THRESHOLD = 0.95
# high_corr_pairs = []

# x_gpu = cp.asarray(X.values).T 
# corr_gpu = cp.abs(cp.corrcoef(x_gpu))
# iu_0_gpu, iu_1_gpu = cp.triu_indices_from(corr_gpu, k=1)
# mask_gpu = corr_gpu[iu_0_gpu, iu_1_gpu] > CORR_THRESHOLD
# idx_0 = iu_0_gpu[mask_gpu].get()
# idx_1 = iu_1_gpu[mask_gpu].get()
# high_corr_pairs = list(zip(cols[idx_0], cols[idx_1]))
# print(f"      ➜ Found {len(high_corr_pairs)} highly correlated pairs.")
# del x_gpu, corr_gpu, iu_0_gpu, iu_1_gpu, mask_gpu
# cp.get_default_memory_pool().free_all_blocks()
# #---------------------------------------------------------------------------------------------#
# #---------------------------------------------------------------------------------------------#
# #---------------------------------------------------------------------------------------------#
# print(f"🔹 [3/4] Running LightGBM 5 times for Stable Importance...")
# SEEDS = [12, 128, 1208, 120804, 12082004]
# importances_df = pd.DataFrame(index=cols)
# for i, seed in enumerate(tqdm(SEEDS)):
#     model = LGBMClassifier(
#         device="gpu", 
#         n_jobs=-1,
#         verbosity=-1,
#         importance_type='gain',
#         n_estimators=2500,    
#         learning_rate=0.02,    
#         num_leaves=31,         
#         max_depth=6,           
#         min_child_samples=500, 
#         subsample=0.7,         
#         subsample_freq=1,      
#         colsample_bytree=0.7,  
#         reg_alpha=0.5,         
#         reg_lambda=1.0,       
#         random_state=seed       
#     )
#     model.fit(X, y)
#     importances_df[f'run_{i}'] = model.feature_importances_
#     del model
#     gc.collect()
# importances_df['mean_gain'] = importances_df.mean(axis=1)
# #---------------------------------------------------------------------------------------------#
# #---------------------------------------------------------------------------------------------#
# #---------------------------------------------------------------------------------------------#
# print("🔹 [4/4] Finalizing drop list based on Average Importance...")
# drop_features = set()
# avg_imp = importances_df['mean_gain'] 
# # Logic 1: Xử lý cặp tương quan (Dựa trên Mean Gain)
# for f1, f2 in high_corr_pairs:
#     if avg_imp[f1] < avg_imp[f2]:
#         drop_features.add(f1)
#     else:
#         drop_features.add(f2)

# # Logic 2: Xử lý Zero Importance 
# zero_imp_features = avg_imp[avg_imp == 0].index.tolist()
# drop_features.update(zero_imp_features)

# # --- LƯU FILE ---
# pd.DataFrame({"drop_features": list(drop_features)}).to_csv("feature_drops.csv", index=False)
# importances_df.sort_values(by='mean_gain', ascending=False)\
#             .reset_index()\
#             .rename(columns={'index': 'feature'})\
#             .to_csv("feature_importance.csv", index=False)
# del X, y, importances_df, avg_imp
# gc.collect()

In [17]:
import pandas as pd
import numpy as np
import cupy as cp
import gc

# ---------------------------------------------------------------------------------------------
# 1. SETUP: LOAD GAIN & DIỆT ZERO IMPORTANCE TRƯỚC
# ---------------------------------------------------------------------------------------------
print("🔹 [1/4] Loading Feature Importance & Removing Zero Gain features...")
imp_df = pd.read_csv("../input/feature-importance/feature_importance.csv")

# Map để tra cứu gain
feature_gain_map = imp_df.set_index('feature')['mean_gain'].fillna(0).to_dict()

# Lấy danh sách feature vô dụng (Gain = 0)
zero_imp_features = imp_df[imp_df['mean_gain'] == 0]['feature'].tolist()

# KHỞI TẠO SET DROP VỚI CÁC FEATURE = 0 LUÔN
drop_features = set(zero_imp_features) 

print(f"   ➜ Identified {len(zero_imp_features)} features with Zero Importance (Will be dropped).")

# ---------------------------------------------------------------------------------------------
# 2. PREPARE DATA (Chỉ lấy cột số để tính corr)
# ---------------------------------------------------------------------------------------------
print("🔹 [2/4] Preparing data subset...")
# Loại bỏ các cột không dùng để tính toán + các cột label
train_subset = df[df["month"].isin(list(range(2, 12)))].drop(columns=["timestamp_dt", "month", "label_5m"])

# Convert float32
X_temp = train_subset.astype(np.float32)
cols = X_temp.columns.to_numpy()

del train_subset
gc.collect()

# ---------------------------------------------------------------------------------------------
# 3. TÍNH CORRELATION (GPU)
# ---------------------------------------------------------------------------------------------
print("🔹 [3/4] Computing Correlation (GPU)...")

# --- ĐIỀU CHỈNH THRESHOLD TẠI ĐÂY ---
# Giảm xuống 0.90 để lọc mạnh tay hơn. 
# Nếu data quá nhiều feature rác, có thể thử 0.85
CORR_THRESHOLD = 0.85 

# Đẩy lên GPU
x_gpu = cp.asarray(X_temp.values).T 
corr_gpu = cp.abs(cp.corrcoef(x_gpu))

# Lấy index tam giác trên
iu_0_gpu, iu_1_gpu = cp.triu_indices_from(corr_gpu, k=1)

# Mask lọc các cặp > Threshold
mask_gpu = corr_gpu[iu_0_gpu, iu_1_gpu] > CORR_THRESHOLD
idx_0 = iu_0_gpu[mask_gpu].get()
idx_1 = iu_1_gpu[mask_gpu].get()

# Giải phóng GPU
del x_gpu, corr_gpu, iu_0_gpu, iu_1_gpu, mask_gpu, X_temp
cp.get_default_memory_pool().free_all_blocks()
gc.collect()

# ---------------------------------------------------------------------------------------------
# 4. GIẢI QUYẾT XUNG ĐỘT (SMART SELECTION) & DROP TRÊN DF
# ---------------------------------------------------------------------------------------------
print(f"🔹 [4/4] Resolving conflicts (Threshold: {CORR_THRESHOLD}) & Dropping...")

count_corr_drops = 0

for i0, i1 in zip(idx_0, idx_1):
    f1 = cols[i0]
    f2 = cols[i1]
    
    # Nếu feature đã nằm trong danh sách loại (do gain=0 hoặc do cặp khác loại rồi) thì bỏ qua
    if f1 in drop_features and f2 in drop_features:
        continue
    
    # Nếu 1 trong 2 đã bị loại, thằng còn lại hiển nhiên được giữ (tạm thời)
    # Ta chỉ quan tâm khi CẢ 2 ĐỀU ĐANG ĐƯỢC GIỮ
    if f1 not in drop_features and f2 not in drop_features:
        gain1 = feature_gain_map.get(f1, 0)
        gain2 = feature_gain_map.get(f2, 0)

        # Drop thằng có gain thấp hơn
        if gain1 < gain2:
            drop_features.add(f1)
            count_corr_drops += 1
        else:
            drop_features.add(f2)
            count_corr_drops += 1
    # Nếu 1 trong 2 đã bị drop thì không cần làm gì thêm

print(f"   ➜ Added {count_corr_drops} features to drop via Correlation.")
print(f"   ➜ Total features to drop: {len(drop_features)}")

# --- THỰC HIỆN DROP TRÊN DF GỐC ---
initial_shape = df.shape
df.drop(columns=list(drop_features), inplace=True, errors='ignore')

# Lưu file backup
pd.DataFrame({"drop_features": list(drop_features)}).to_csv("final_drop_list.csv", index=False)

print(f"✅ DONE! DF Updated.")
print(f"   Before: {initial_shape}")
print(f"   After:  {df.shape}")

🔹 [1/4] Loading Feature Importance & Removing Zero Gain features...
   ➜ Identified 37 features with Zero Importance (Will be dropped).
🔹 [2/4] Preparing data subset...
🔹 [3/4] Computing Correlation (GPU)...
🔹 [4/4] Resolving conflicts (Threshold: 0.85) & Dropping...
   ➜ Added 550 features to drop via Correlation.
   ➜ Total features to drop: 587
✅ DONE! DF Updated.
   Before: (94752, 3506)
   After:  (94752, 2919)


In [18]:
df = df.sort_values("timestamp_dt").reset_index(drop=True)
df = df.dropna(subset=["label_5m"]).reset_index(drop=True)
df["timestamp_dt"] = df["timestamp_dt"].dt.tz_localize(None)

In [19]:
import pandas as pd
import numpy as np
import gc
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report, f1_score

# =========================
# CẤU HÌNH
# =========================
LABEL_COL = "label_5m"
THRESHOLDS_TO_TEST = [0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80]

evaluation_results = []

print("\n==============================================")
print("🚀 TRAIN 1 LẦN (T1–T11) | TEST THÁNG 12")
print("==============================================")

# =========================
# 1. CHUẨN BỊ DỮ LIỆU
# =========================
if not np.issubdtype(df['timestamp_dt'].dtype, np.datetime64):
    df['timestamp_dt'] = pd.to_datetime(df['timestamp_dt'])

# Split data
train_df = df[df["month"].isin(range(1, 12))].copy()   # T1 → T11
test_df  = df[df["month"] == 12].copy()                # Tháng 12

# Drop columns
drop_cols = ["timestamp_dt", "month", LABEL_COL]
if "day" in train_df.columns:
    drop_cols.append("day")

X_train = train_df.drop(columns=drop_cols, errors="ignore").astype("float32")
y_train = train_df[LABEL_COL]

drop_cols_test = ["timestamp_dt", "month", LABEL_COL]
if "day" in test_df.columns:
    drop_cols_test.append("day")

X_test = test_df.drop(columns=drop_cols_test, errors="ignore").astype("float32")
y_test = test_df[LABEL_COL]

print(f"Train samples: {len(X_train):,}")
print(f"Test samples : {len(X_test):,}")

# =========================
# 2. TRAIN MODEL (1 LẦN)
# =========================
model = LGBMClassifier(
    device="gpu",
    n_jobs=-1,
    objective="multiclass",
    n_estimators=2500,
    learning_rate=0.02,
    num_leaves=45,
    max_depth=8,
    min_child_samples=100,
    class_weight={0: 1, 1: 1.5, 2: 1.5},
    random_state=42,
    verbosity=-1
)

model.fit(X_train, y_train)

# Predict probability
y_prob = model.predict_proba(X_test)

# =========================
# 3. SCAN THRESHOLD (THÁNG 12)
# =========================
# =========================
# 3. SCAN THRESHOLD (THÁNG 12)
# =========================
for th in THRESHOLDS_TO_TEST:
    y_pred = np.zeros(len(y_test), dtype=np.int8)

    mask_down = (y_prob[:, 1] >= th) & (y_prob[:, 1] > y_prob[:, 2])
    mask_up   = (y_prob[:, 2] >= th) & (y_prob[:, 2] > y_prob[:, 1])

    y_pred[mask_down] = 1
    y_pred[mask_up]   = 2

    report_dict = classification_report(
        y_test, y_pred, output_dict=True, zero_division=0
    )

    report_text = classification_report(
        y_test, y_pred, digits=4, zero_division=0
    )

    print("\n" + "="*60)
    print(f"📊 CLASSIFICATION REPORT | THRESHOLD = {th:.2f}")
    print("="*60)
    print(report_text)

    evaluation_results.append({
        "threshold": th,

        "prec_down": report_dict.get('1', {}).get('precision', 0),
        "rec_down":  report_dict.get('1', {}).get('recall', 0),
        "prec_up":   report_dict.get('2', {}).get('precision', 0),
        "rec_up":    report_dict.get('2', {}).get('recall', 0),

        "macro_f1": f1_score(y_test, y_pred, average='macro'),

        "count_down": int(np.sum(y_pred == 1)),
        "count_up":   int(np.sum(y_pred == 2)),
        "total_signals": int(np.sum((y_pred == 1) | (y_pred == 2)))
    })




🚀 TRAIN 1 LẦN (T1–T11) | TEST THÁNG 12
Train samples: 87,264
Test samples : 7,488


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.



📊 CLASSIFICATION REPORT | THRESHOLD = 0.35
              precision    recall  f1-score   support

           0     0.5230    0.3476    0.4176      3340
           1     0.3974    0.5339    0.4556      2053
           2     0.4183    0.5012    0.4560      2095

    accuracy                         0.4416      7488
   macro avg     0.4462    0.4609    0.4431      7488
weighted avg     0.4593    0.4416    0.4388      7488


📊 CLASSIFICATION REPORT | THRESHOLD = 0.40
              precision    recall  f1-score   support

           0     0.5048    0.5335    0.5188      3340
           1     0.4334    0.4359    0.4347      2053
           2     0.4538    0.4100    0.4308      2095

    accuracy                         0.4722      7488
   macro avg     0.4640    0.4598    0.4614      7488
weighted avg     0.4710    0.4722    0.4711      7488


📊 CLASSIFICATION REPORT | THRESHOLD = 0.45
              precision    recall  f1-score   support

           0     0.4886    0.6838    0.5699      33

In [20]:
sdfadsf

NameError: name 'sdfadsf' is not defined

In [ ]:
ádfsfd

In [ ]:
feature_importance_ranks

In [ ]:
all_dropped_features = [f for s in feature_drop_sets for f in s]

# Đếm số lần mỗi feature bị đề nghị drop
drop_counts = Counter(all_dropped_features)
total_folds = len(feature_drop_sets)

# Chuyển sang DataFrame để dễ vẽ
df_votes = pd.DataFrame.from_dict(drop_counts, orient='index', columns=['votes'])
df_votes['vote_ratio'] = df_votes['votes'] / total_folds

print(f"📊 Tổng số feature từng bị đề nghị drop ít nhất 1 lần: {len(df_votes)}")
print(f"🔢 Tổng số Fold đã chạy: {total_folds}")

# ===== 2. VẼ HISTOGRAM =====
plt.figure(figsize=(12, 6))
sns.set_style("whitegrid")

# Vẽ biểu đồ cột
ax = sns.histplot(
    data=df_votes, 
    x='votes', 
    bins=range(1, total_folds + 2), 
    discrete=True, 
    color='#e74c3c', # Màu đỏ thể hiện cảnh báo/drop
    shrink=0.8       # Làm cột nhỏ lại chút cho đẹp
)

# Trang trí biểu đồ
plt.title('PHÂN PHỐI TẦN SUẤT BỊ DROP CỦA FEATURES (EXPANDING WINDOW)', fontsize=14, fontweight='bold', pad=20)
plt.xlabel('Số lượng Fold đề xuất Drop (Càng cao = Càng Rác)', fontsize=12)
plt.ylabel('Số lượng Feature', fontsize=12)
plt.xticks(range(1, total_folds + 1)) # Đánh số trục X từ 1 đến max fold

# Thêm số liệu trên đầu mỗi cột
for p in ax.patches:
    height = int(p.get_height())
    if height > 0:
        ax.annotate(f'{height}', 
                    (p.get_x() + p.get_width() / 2., height), 
                    ha='center', va='bottom', 
                    fontsize=11, color='black', fontweight='bold')

# Vẽ đường Threshold gợi ý (ví dụ: 70%)
threshold_val = int(total_folds * 0.7)
plt.axvline(x=threshold_val + 0.5, color='blue', linestyle='--', linewidth=2, label=f'Threshold Cắt bỏ (>{int(threshold_val)})')
plt.legend()

plt.show()

# ===== 3. XUẤT RA DANH SÁCH CUỐI CÙNG =====
# Ngưỡng chọn: Loại bỏ nếu bị vote drop trong > 70% số fold
THRESHOLD_RATIO = 0.7 
features_to_remove = df_votes[df_votes['vote_ratio'] > THRESHOLD_RATIO].index.tolist()

print("\n✂️ KẾT QUẢ QUYẾT ĐỊNH:")
print(f"   - Ngưỡng cắt bỏ: > {THRESHOLD_RATIO*100}% số fold ({int(total_folds * THRESHOLD_RATIO)} folds)")
print(f"   - Số feature cần loại bỏ vĩnh viễn: {len(features_to_remove)}")
print(f"   - Số feature 'an toàn' (hoặc ít bị drop): {len(df_votes) - len(features_to_remove)} (trong số những cái bị flag)")

##### Adversarial validation

##### Mean rank feature important

In [ ]:
all_ranks_df = pd.concat(feature_importance_ranks, axis=1)
mean_rank = all_ranks_df.mean(axis=1).sort_values()

##### Recursive feature elimination

In [ ]:
# import numpy as np
# import pandas as pd
# from sklearn.feature_selection import mutual_info_classif

# # =====================================================
# # 0. CONFIG
# # =====================================================

# TIME_COL  = "timestamp_dt"
# LABEL_COL = "label_5m"

# CORR_TH  = 0.95      # ngưỡng corr feature-feature
# EPS_CORR = 0.01      # ngưỡng so sánh corr(feature, label)
# EPS_MI   = 1e-4      # ngưỡng so sánh mutual information

# # =====================================================
# # 1. PREPARE DATA
# # =====================================================

# # sort time để đảm bảo nhất quán
# df = df.sort_values(TIME_COL).reset_index(drop=True)

# # chỉ lấy cột số, loại timestamp & label
# EXCLUDE = {TIME_COL, LABEL_COL}
# num_cols = [
#     c for c in df.columns
#     if c not in EXCLUDE and np.issubdtype(df[c].dtype, np.number)
# ]

# X = df[num_cols]
# y = df[LABEL_COL]

# print(f"🔢 Initial numeric features: {X.shape[1]}")

# # =====================================================
# # 2. FEATURE–FEATURE CORRELATION
# # =====================================================

# corr_matrix = X.corr().abs()

# # chỉ lấy tam giác trên (tránh trùng lặp)
# upper = corr_matrix.where(
#     np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
# )

# pairs = []
# for col in upper.columns:
#     high_corr = upper[col][upper[col] >= CORR_TH]
#     for row_name, corr_val in high_corr.items():
#         pairs.append((row_name, col, corr_val))

# pairs = sorted(pairs, key=lambda x: x[2], reverse=True)

# print(f"🔗 High-correlation pairs (>= {CORR_TH}): {len(pairs)}")

# # =====================================================
# # 3. FEATURE–LABEL METRICS
# # =====================================================

# # Corr(feature, label)
# corr_with_label = X.corrwith(y).abs()

# # Variance
# var = X.var()

# # Mutual Information (phi tuyến)
# mi = pd.Series(
#     mutual_info_classif(
#         X.fillna(0),
#         y,
#         discrete_features=False,
#         random_state=42
#     ),
#     index=X.columns
# )

# # =====================================================
# # 4. FEATURE PRUNING – LEVEL 3
# # =====================================================

# to_drop = set()
# active  = set(X.columns)

# fmt = (
#     "{:<45} | {:<45} | "
#     "Corr={:<6} | "
#     "Corr_y(A)={:<6} Corr_y(B)={:<6} | "
#     "MI(A)={:<7} MI(B)={:<7} | "
#     "Winner={:<45} | Reason={}"
# )

# header = fmt.format(
#     "Feature A", "Feature B",
#     "", "", "",
#     "", "",
#     "", ""
# )

# print("\n" + header)
# print("-" * len(header))

# for a, b, corr_ab in pairs:

#     # nếu đã bị loại thì bỏ qua
#     if a not in active or b not in active:
#         continue

#     ca, cb = corr_with_label[a], corr_with_label[b]
#     mia, mib = mi[a], mi[b]
#     vara, varb = var[a], var[b]

#     # ---------- LEVEL 3 DECISION ----------
#     if abs(ca - cb) > EPS_CORR:
#         reason = "corr(label)"
#         winner, loser = (a, b) if ca > cb else (b, a)

#     elif abs(mia - mib) > EPS_MI:
#         reason = "mutual_info"
#         winner, loser = (a, b) if mia > mib else (b, a)

#     else:
#         reason = "variance"
#         winner, loser = (a, b) if vara > varb else (b, a)
#     # ------------------------------------

#     to_drop.add(loser)
#     active.remove(loser)

#     print(fmt.format(
#         a, b,
#         f"{corr_ab:.3f}",
#         f"{ca:.3f}", f"{cb:.3f}",
#         f"{mia:.4f}", f"{mib:.4f}",
#         winner,
#         reason
#     ))

# # =====================================================
# # 5. DROP FEATURES
# # =====================================================

# df_pruned = df.drop(columns=list(to_drop))

# print("-" * len(header))
# print(f"❌ Dropped features: {len(to_drop)}")
# print(f"✅ Remaining features: {len(active)}")
# print(f"📊 Final dataframe shape: {df_pruned.shape}")


Filter theo unique, variance, corr, và MI

### Train & Test Model

Filter feature theo model, chuẩn hóa và detact outlier trong lúc train và test để tránh leakage

Nhóm theo ngày
EDA vẽ biểu đồ, tìm ra dữ liệu thiếu và bất thường --> resample, smooth, rolling mean 
Plot cho cái 5m
CSV -->
Nhóm theo từng tháng/tuần/ngày --> Gía trị trung bình theo từng tháng
Đường trung bình --> Upper band --> Lower band
Lọc bằng corr, tạo lag feature, tạo derive feature, lọc bằng model, tạo biến target.

# WALK-FORWARD VALIDATION

In [ ]:
# for train_window in range(2, 12): 
#     train_months = list(range(1, train_window + 1))
#     test_month = train_months[-1] + 1
#     while test_month <= 12:
#         print(f"Train months: {train_months} | Test month: {test_month}")
#         train_months = [m + 1 for m in train_months]
#         test_month = train_months[-1] + 1